### Phase 1: Environment & Dependency Setup
This section initializes the necessary libraries and BigQuery clients required for processing the NBA dataset.

In [ ]:
import os

# Retrieve BigQuery configuration from environment variables for reproducibility
PROJECT_ID = os.environ.get("BQ_PROJECT_ID", "mapping-nba-fouls")
DATASET_ID = os.environ.get("BQ_DATASET_ID", "capstone_project")

print(f"Active Project ID: {PROJECT_ID}")
print(f"Active Dataset ID: {DATASET_ID}")

In [ ]:
import pandas as pd
import requests
import tarfile
import os
import io
import numpy as np
from google.cloud import bigquery


### Phase 2: Multi-Season Data Ingestion
We aggregate regular season (`2YYYY`) and playoff (`4YYYY`) data from 2019 through 2024. This process includes decompressing `.tar.xz` archives and injecting metadata for traceability.

In [ ]:

# Configure data sources and destination lists
SEASONS = ["2019", "2020", "2021", "2022", "2023", "2024"]
BASE_URL = "https://raw.githubusercontent.com/shufinskiy/nba_data/main/datasets"
DATA_TYPE = "datanba" # options: 'datanba' (regular) or 'datanba_po' (playoffs)

all_dfs = []

def download_and_extract(season, data_prefix):
    file_name = f"{data_prefix}_{season}.tar.xz"
    url = f"{BASE_URL}/{file_name}"

    print(f"Fetching {file_name}...")
    response = requests.get(url)

    if response.status_code == 200:
        # Extract .tar.xz in memory or to temp disk
        with tarfile.open(fileobj=io.BytesIO(response.content), mode="r:xz") as tar:
            # Assumes one CSV inside with a similar name
            for member in tar.getmembers():
                if member.name.endswith(".csv"):
                    f = tar.extractfile(member)
                    temp_df = pd.read_csv(f)

                    # NBA API Season ID logic: Regular (2YYYY) or Playoffs (4YYYY)
                    year_start = season.split('-')[0]
                    prefix = "2" if "po" not in data_prefix else "4"
                    temp_df['season_id'] = f"{prefix}{year_start}"

                    # --- NEW METADATA ---
                    temp_df['source_file'] = url # Tracking origin

                    return temp_df
    else:
        print(f"Failed to download {season}. Status: {response.status_code}")
        return None

# Loop through both Regular Season and Playoffs
for season in SEASONS:
    # Regular Season
    df_rg = download_and_extract(season, "datanba")
    if df_rg is not None: all_dfs.append(df_rg)

    # Playoffs
    df_po = download_and_extract(season, "datanba_po")
    if df_po is not None: all_dfs.append(df_po)

# Combine all into one master DataFrame
raw_df = pd.concat(all_dfs, ignore_index=True)
print(f"Total rows ingested: {len(raw_df)}")
print(raw_df.head(5))
print(raw_df.tail(5))

### Phase 3: Data Quality & Standardization
To ensure downstream analytical integrity, we enforce schema types (String for IDs, Integer for coordinates) and generate a data quality report to identify missing values across seasons.

In [ ]:
def clean_nba_data(df):
    cleaned = df.copy()

    # Inject a single UTC timestamp for the ingestion batch
    cleaned['ingested_at'] = pd.Timestamp.now(tz='UTC')

    # Remove exact duplicate records
    initial_count = len(cleaned)
    cleaned = cleaned.drop_duplicates()
    if initial_count != len(cleaned):
        print(f"Removed {initial_count - len(cleaned)} duplicate rows.")

    # 1. Convert wall clock string to UTC timestamp
    print("Converting wall_clock to TIMESTAMP (ISO8601)...")
    cleaned['wallclk'] = pd.to_datetime(cleaned['wallclk'], errors='coerce', format='ISO8601', utc=True)

    # 2. Clock Math & Display
    def parse_seconds(val):
        if pd.isna(val) or val == "" or str(val) == "0": return 0.0
        val = str(val).strip()
        if ":" not in val:
            try: return float(val)
            except: return 0.0
        parts = val.split(":")
        return (int(parts[0]) * 60) + float(parts[1])

    cleaned['cl_seconds'] = cleaned['cl'].apply(parse_seconds)
    cleaned['cl_display'] = cleaned['cl_seconds'].apply(lambda x: f"{int(x // 60):02d}:{x % 60:04.1f}")

    # 3. ID and Type Conversions (STRING)
    id_cols = ['evt', 'opid', 'tid', 'pid', 'epid', 'oftid', 'GAME_ID', 'mtype', 'etype', 'season_id', 'ord']
    for col in id_cols:
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].fillna(-1).astype(float).astype(int).astype(str)
            cleaned[col] = cleaned[col].replace(['-1', '-1.0'], None).str.replace(r'\.0$', '', regex=True)

    # 4. Numeric Conversions (Nullable Integer handling Missing Vals)
    int_cols = ['locX', 'locY', 'opt1', 'opt2', 'opt3', 'opt4', 'PERIOD', 'hs', 'vs', 'pts']
    for col in int_cols:
        if col in cleaned.columns:
            cleaned[col] = pd.to_numeric(cleaned[col], errors='coerce').astype('Int64')

    # 5. Map columns to standard warehouse schema
    rename_map = {
        'evt': 'event_num',
        'wallclk': 'wall_clock',
        'cl_seconds': 'game_clock_seconds',
        'cl_display': 'game_clock',
        'de': 'description',
        'locX': 'loc_x',
        'locY': 'loc_y',
        'opt1': 'option_1',
        'opt2': 'option_2',
        'opt3': 'option_3',
        'opt4': 'option_4',
        'mtype': 'msg_type',
        'etype': 'event_type',
        'opid': 'opp_player_id',
        'tid': 'team_id',
        'pid': 'player_id',
        'hs': 'home_score',
        'vs': 'visitor_score',
        'epid': 'extra_player_id',
        'oftid': 'offense_team_id',
        'ord': 'sequence_num',
        'pts': 'points',
        'PERIOD': 'period',
        'GAME_ID': 'game_id'
    }

    cleaned = cleaned.rename(columns=rename_map)

    # Drop the original 'cl' as we now have seconds and display versions
    if 'cl' in cleaned.columns:
        cleaned = cleaned.drop(columns=['cl'])

    return cleaned

clean_df = clean_nba_data(raw_df)
print("Standardized Columns:", clean_df.columns.tolist())

In [ ]:


def run_data_quality_checks(df):
    """Generates a summary of data quality issues before cleaning."""
    print("--- Initial Data Quality Report ---")
    total_rows = len(df)

    # 1. Missing Values
    missing = df.isnull().sum()
    missing_pct = (missing / total_rows) * 100
    missing_report = pd.DataFrame({'Missing Count': missing, 'Percent (%)': missing_pct})
    print("\nMissing values per column:")
    display(missing_report[missing_report['Missing Count'] > 0])

    # 2. Duplicate Records
    duplicate_count = df.duplicated().sum()
    print(f"\nExact duplicate rows found: {duplicate_count}")

    return duplicate_count


In [ ]:
# 1. Run the quality report first
duplicates = run_data_quality_checks(clean_df)

**Data Quality Assessment:**
The absence of duplicates and complete `loc_x` / `loc_y` coordinate data indicates strong spatial data quality. Expected missing values in `opp_player_id` (`opid`) are noted and will be handled during subsequent analytical transformations.

In [ ]:
# Verify all columns exist
print(f"Total columns: {len(clean_df.columns)}")
print(list(clean_df.columns))

**Validation Complete:** The dataset is standardized and ready for warehouse ingestion.

### Phase 4: BigQuery Warehouse Loading
The cleaned dataset is loaded into a clustered BigQuery table. Clustering on `season_id` and `event_type` optimizes query performance for foul-specific analysis.

In [ ]:

# 1. Configuration
TABLE_NAME = "raw_shufinskiy_datanba"
TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"

client = bigquery.Client(project=PROJECT_ID)

# 2. Define logically ordered BigQuery schema for intuitive exploration
schema = [
    # --- CONTEXT & TRACEABILITY ---
    bigquery.SchemaField("season_id", "STRING"),
    bigquery.SchemaField("game_id", "STRING"),
    bigquery.SchemaField("sequence_num", "STRING"),
    bigquery.SchemaField("source_file", "STRING"),    # Traceability
    bigquery.SchemaField("ingested_at", "TIMESTAMP"), # Traceability

    # --- TIME ---
    bigquery.SchemaField("period", "INTEGER"),
    bigquery.SchemaField("game_clock", "STRING"),
    bigquery.SchemaField("game_clock_seconds", "FLOAT64"),
    bigquery.SchemaField("wall_clock", "TIMESTAMP"),

    # --- EVENT ---
    bigquery.SchemaField("event_num", "STRING"),
    bigquery.SchemaField("event_type", "STRING"),
    bigquery.SchemaField("msg_type", "STRING"),
    bigquery.SchemaField("description", "STRING"),

    # --- SPATIAL ---
    bigquery.SchemaField("loc_x", "INTEGER"),
    bigquery.SchemaField("loc_y", "INTEGER"),

    # --- PARTICIPANTS ---
    bigquery.SchemaField("player_id", "STRING"),
    bigquery.SchemaField("team_id", "STRING"),
    bigquery.SchemaField("opp_player_id", "STRING"),
    bigquery.SchemaField("extra_player_id", "STRING"),
    bigquery.SchemaField("offense_team_id", "STRING"),

    # --- GAME STATE ---
    bigquery.SchemaField("points", "INTEGER"),
    bigquery.SchemaField("home_score", "INTEGER"),
    bigquery.SchemaField("visitor_score", "INTEGER"),

    # --- METADATA/OPTIONS ---
    bigquery.SchemaField("option_1", "INTEGER"),
    bigquery.SchemaField("option_2", "INTEGER"),
    bigquery.SchemaField("option_3", "INTEGER"),
    bigquery.SchemaField("option_4", "INTEGER"),
]

# 3. Optimize target table by clustering on highly-queried analytical dimensions
job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition="WRITE_TRUNCATE", # Clean overwrite for MVP
    # Optimized for ETL: Pulling specific event types (like fouls) for specific seasons
    clustering_fields=["season_id", "event_type", "msg_type"]
)

# 4. Execute Load
print(f"🚀 Starting load of {len(clean_df)} rows to {TABLE_ID}...")
load_job = client.load_table_from_dataframe(clean_df, TABLE_ID, job_config=job_config)
load_job.result() # Wait for completion

print(f"✅ Load Complete.")

# 5. Professional Audit Query
# This verifies the integrity of the data immediately after load.
audit_query = f"""
SELECT
    season_id,
    source_file,
    FORMAT_TIMESTAMP('%Y-%m-%d %H:%M:%S', MAX(ingested_at)) as last_ingest,
    COUNT(*) as total_rows,
    COUNT(DISTINCT game_id) as games_found,
    COUNTIF(wall_clock IS NULL) as missing_wallclock,
    COUNTIF(loc_x IS NULL) as missing_coords
FROM `{TABLE_ID}`
GROUP BY 1, 2
ORDER BY 1 DESC
"""

print("\n--- INGESTION AUDIT REPORT ---")
audit_results = client.query(audit_query).to_dataframe()
display(audit_results)